# 1-Day Weather Forecast Scaffold

This notebook is a small starter project for predicting **tomorrow's maximum temperature** for a single location.

We'll keep the first version intentionally simple:
- Pull daily historical weather data
- Build a few lag and rolling features
- Compare a naive baseline against a first regression model
- Leave room to improve the theory later

## Setup

If you need the packages locally, install them with:

```bash
pip install pandas numpy matplotlib scikit-learn meteostat
```

In [ ]:
from datetime import date
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

from meteostat import Daily, Point

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

## Project Configuration

Change the coordinates or date range if you want a different location.

In [ ]:
LOCATION_NAME = "San Francisco, CA"
LATITUDE = 37.7749
LONGITUDE = -122.4194
ALTITUDE = 16

START_DATE = date(2018, 1, 1)
END_DATE = date(2025, 12, 31)

LOCATION_NAME, START_DATE, END_DATE

## Pull Historical Daily Data

In [ ]:
location = Point(LATITUDE, LONGITUDE, ALTITUDE)
weather = Daily(location, START_DATE, END_DATE).fetch().reset_index()

weather = weather.rename(columns={"time": "date"}).sort_values("date").reset_index(drop=True)

core_columns = ["date", "tmin", "tmax", "temp", "prcp", "wspd", "pres"]
available_columns = [col for col in core_columns if col in weather.columns]
weather = weather[available_columns].copy()

weather.head()

In [ ]:
weather.info()
weather.describe(include="all")

## Feature Engineering

We'll predict `target_tmax_next_day` using only information that would be available before tomorrow happens.

In [ ]:
df = weather.copy()

numeric_inputs = [col for col in ["tmin", "tmax", "temp", "prcp", "wspd", "pres"] if col in df.columns]

for col in numeric_inputs:
    df[f"{col}_lag1"] = df[col].shift(1)
    df[f"{col}_rolling3"] = df[col].shift(1).rolling(3).mean()
    df[f"{col}_rolling7"] = df[col].shift(1).rolling(7).mean()

df["day_of_year"] = df["date"].dt.dayofyear
df["month"] = df["date"].dt.month
df["target_tmax_next_day"] = df["tmax"].shift(-1)

model_df = df.dropna().reset_index(drop=True)
model_df.head()

## Train / Test Split

This uses a simple time-based split. We do **not** shuffle weather data.

In [ ]:
feature_cols = [
    col for col in model_df.columns
    if col not in {"date", "target_tmax_next_day"} and pd.api.types.is_numeric_dtype(model_df[col])
]

split_idx = int(len(model_df) * 0.8)

train_df = model_df.iloc[:split_idx].copy()
test_df = model_df.iloc[split_idx:].copy()

X_train = train_df[feature_cols]
y_train = train_df["target_tmax_next_day"]
X_test = test_df[feature_cols]
y_test = test_df["target_tmax_next_day"]

print(f"Training rows: {len(train_df)}")
print(f"Testing rows:  {len(test_df)}")
print(f"Feature count: {len(feature_cols)}")

## Baseline vs First Model

Baseline idea: tomorrow's max temperature will be the same as today's max temperature.

In [ ]:
baseline_pred = test_df["tmax"].to_numpy()

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)
model_pred = model.predict(X_test)

results = pd.DataFrame(
    {
        "date": test_df["date"],
        "actual": y_test,
        "baseline_pred": baseline_pred,
        "model_pred": model_pred,
    }
)

results.head()

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred, squared=False),
    }


metrics_df = pd.DataFrame(
    [
        {"model": "baseline_same_as_today", **regression_metrics(y_test, baseline_pred)},
        {"model": "ridge_regression", **regression_metrics(y_test, model_pred)},
    ]
)

metrics_df.sort_values("mae")

In [ ]:
coef_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "coefficient": model.coef_,
    }
).sort_values("coefficient", key=np.abs, ascending=False)

coef_df.head(15)

In [ ]:
plot_df = results.tail(90).copy()

plt.figure(figsize=(14, 5))
plt.plot(plot_df["date"], plot_df["actual"], label="actual", linewidth=2)
plt.plot(plot_df["date"], plot_df["baseline_pred"], label="baseline", alpha=0.8)
plt.plot(plot_df["date"], plot_df["model_pred"], label="ridge", alpha=0.8)
plt.title(f"{LOCATION_NAME}: next-day max temperature prediction")
plt.xlabel("date")
plt.ylabel("temperature")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Next Theory Questions

Once this scaffold is running, useful next improvements are:
- Better train/validation/test windows by year
- Seasonal encoding with sine/cosine instead of raw month/day values
- Rain prediction as a separate classification problem
- More expressive models like random forests or gradient boosting
- Forecasting from hourly data instead of daily aggregates
- Comparing station data vs model/reanalysis data